# 03 · RAG with privacy
**The limitation this fixes:** the agent can't answer from our leaflets or check-in notes, and it would happily echo identifiers.

| Feature | Framework piece |
|---|---|
| Load, chunk, embed, index | `Document`, `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `InMemoryVectorStore` |
| Retrieval | just another `@tool` |
| Show sources | `response_format="content_and_artifact"` |
| Patient A can't see patient B's notes | filter built from `runtime.context` |
| Hide MRNs and phone numbers | `PIIMiddleware` |
| Audit log | `@wrap_tool_call` middleware |

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far

In [ ]:
from common import (CareContext, SYSTEM_PROMPT, DATA, get_care_plan, log_reading, log_symptom,
                    get_recent_readings, escalate_to_nurse, dose_guard, run_eval, ask, resume, show_state)
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

## 1. Load the documents
Leaflets are **public**. Check-in notes are **private** and tagged with the patient they belong to.

In [ ]:
from langchain_core.documents import Document

docs = [Document(p.read_text(), metadata={"source": p.name, "scope": "public"})
        for p in sorted((DATA / "leaflets").glob("*.md"))]
docs += [Document(p.read_text(), metadata={"source": p.name, "scope": "private", "patient_token": p.stem})
         for p in sorted((DATA / "notes").glob("*.md"))]

assert docs, "data/ is empty"
[d.metadata for d in docs]

## 2. Chunk, embed, index

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
len(chunks)

## 3. Retrieval is just a tool
The **filter** is built from `runtime.context`, inside the tool. The model only gets to choose the `query`.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool(response_format="content_and_artifact")
def search_documents(query: str, runtime: ToolRuntime[CareContext]):
    """Search approved patient leaflets and this patient's own check-in notes."""
    me = runtime.context.patient_token

    def allowed(doc):   # public leaflets + only MY notes
        return doc.metadata["scope"] == "public" or doc.metadata.get("patient_token") == me

    found = vector_store.similarity_search(query, k=3, filter=allowed)
    text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in found)
    return text or "Nothing found in the documents.", [d.metadata for d in found]   # (for the model, for us)

## 4. Hide identifiers: `PIIMiddleware`
We only supply the regex for *our* ID formats. LangChain does the redacting on input, output and tool results.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

pii = [
    PIIMiddleware("mrn", detector=r"MRN-\d{6}", strategy="redact",          # CUSTOM: our MRN format
                  apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
    PIIMiddleware("phone", detector=r"\b555-\d{4}\b", strategy="redact",    # CUSTOM: our phone format
                  apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
]

## 5. Audit log: `@wrap_tool_call`
Wraps every tool call: who, which tool, which patient, when.

In [ ]:
from datetime import datetime
from langchain.agents.middleware import wrap_tool_call

AUDIT_LOG = []

@wrap_tool_call
def audit_log(request, handler):
    ctx = request.runtime.context
    AUDIT_LOG.append({"time": datetime.now().strftime("%H:%M:%S"), "user": ctx.user_id,
                      "patient": ctx.patient_token, "tool": request.tool_call["name"]})
    return handler(request)   # run the actual tool

## Putting it together

In [ ]:
RAG_PROMPT = SYSTEM_PROMPT + """
- For how-to questions, or what the doctor noted or planned at past check-ins,
  use search_documents and cite the [source].
- If the documents don't contain the answer, say "That's not in my documents." Don't guess."""

tools = [get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse, search_documents]
agent = create_agent(MODEL, tools=tools, system_prompt=RAG_PROMPT, context_schema=CareContext,
                     checkpointer=InMemorySaver(),
                     middleware=[dose_guard, *pii, audit_log,
                                 HumanInTheLoopMiddleware(interrupt_on={"escalate_to_nurse": True})])

ask(agent, "How should I measure my blood pressure at home?", thread="rag")

🧾 **Receipt:** the artifact holds the sources of the chunks the answer came from (the model never sees it).

In [ ]:
last_tool_message = [m for m in agent.get_state({"configurable": {"thread_id": "rag"}}).values["messages"]
                     if m.type == "tool"][-1]
last_tool_message.artifact

## ❌ Failure case: Sara asks for Omar's notes
The retriever literally cannot return them: the filter came from Sara's context.

In [ ]:
ask(agent, "What do Omar Demo's check-in notes say about his knee?", thread="snoop")
show_state(agent, thread="snoop")

## PII: the MRN never reaches the model
(The tool result also contained Sara's MRN and phone from her notes. Look for `[REDACTED_...]`.)

In [ ]:
ask(agent, "My MRN is MRN-100001. What did my doctor plan at my last check-in?", thread="pii")
show_state(agent, thread="pii")

🧾 **Receipt:** the audit log.

In [ ]:
AUDIT_LOG

## Rerun the benchmark
Same dataset, new experiment. Compare it against `02-baseline` in LangSmith.

In [ ]:
results = run_eval(agent, "03-rag")
results.to_pandas()[["inputs.question", "feedback.used_right_tool",
                     "feedback.mentions_key_fact", "feedback.paused_when_expected"]]

## Where this lives in the backend
- `app/rag.py: loading, chunking, the vector store and `search_documents``
- `app/middleware.py: `pii_middleware()`, `audit_log``

Same primitives, just split into files. The notebook never imports `app/`.